# 個人資訊問答系統 (RAG Application)

### 執行前檢查清單：
1. **啟動 Ollama**：請確保您的 Ollama 服務正在運行 (`ollama serve`)。
2. **下載模型**：
   - 語意嵌入模型：`ollama pull nomic-embed-text` (預設)
   - 對話模型：`ollama pull qwen2.5` (預設)
3. **檢查檔案**：確保 `personal_information` 資料夾內有您的 PDF 文件。

In [ ]:
# 執行主程式並開啟對話視窗 (若只想看視覺化，可跳過此格)
%run main.py

## 繪圖工具與資料準備
此儲存格負責匯入模組、定義工具函式，並**自動連線向量資料庫**以提取繪圖所需的原始資料。執行此格後，即可直接進行 2D/3D 視覺化。

In [ ]:
import os
import yaml
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

load_dotenv()

def wrap_text(text, width=40):
    """為 Plotly tooltip 加入換行符號"""
    if not text: return ""
    clean_text = text.replace('\n', ' ').strip()
    return "<br>".join([clean_text[i:i+width] for i in range(0, len(clean_text), width)])

# 檢查是否需要手動連線資料庫 (若未執行 main.py)
if 'vectorstore' not in locals() or vectorstore is None:
    print("偵測到未執行 main.py，正在根據 config.yaml 手動連線向量資料庫...")
    with open("config.yaml", "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    
    db_name = config["vector_db"]["db_name"]
    embed_provider = config["providers"]["embed_provider"]
    ollama_model = config["providers"]["ollama_embed_model"]

    if embed_provider == "openai":
        embeddings = OpenAIEmbeddings()
    else:
        embeddings = OllamaEmbeddings(model=ollama_model)
    
    if os.path.exists(db_name):
        vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)
        print(f"成功連線至資料庫：{db_name}")
    else:
        print(f"錯誤：找不到資料庫資料夾 '{db_name}'。請先執行 main.py 以建立資料庫。")
        vectorstore = None

if vectorstore:
    print("正在從資料庫提取向量與文件資料...")
    collection_data = vectorstore._collection.get(include=['embeddings', 'metadatas', 'documents'])
    embeddings_data = np.array(collection_data['embeddings'])
    metadatas = collection_data['metadatas']
    documents = collection_data['documents']
    
    if len(embeddings_data) > 0:
        sources = [m.get('source', 'Unknown') for m in metadatas]
        unique_sources = list(set(sources))
        print(f"資料準備就緒！共有 {len(embeddings_data)} 個片段，來自 {len(unique_sources)} 個檔案。")
    else:
        print("警告：資料庫為空。")
else:
    embeddings_data = []

## 向量資料庫視覺化 (2D t-SNE)
利用 t-SNE 將高維向量投影到 2D 平面。

In [ ]:
if len(embeddings_data) > 0:
    print("正在進行 2D 降維運算 (t-SNE)...")
    tsne_2d = TSNE(n_components=2, perplexity=min(30, len(embeddings_data)-1), init='pca', learning_rate='auto', random_state=42)
    embeddings_2d = tsne_2d.fit_transform(embeddings_data)

    fig = go.Figure()
    for i, source in enumerate(unique_sources):
        mask = np.array([s == source for s in sources])
        fig.add_trace(go.Scatter(
            x=embeddings_2d[mask, 0],
            y=embeddings_2d[mask, 1],
            mode='markers',
            name=source,
            text=[f"<b>File:</b> {source}<br><b>Content:</b><br>{wrap_text(documents[j][:200])}..." for j in np.where(mask)[0]],
            hoverinfo='text',
            marker=dict(size=8)
        ))
    
    fig.update_layout(
        title=f"Vector Database 2D Visualization (Total Chunks: {len(embeddings_data)})",
        template="plotly_white", width=900, height=650,
        hoverlabel=dict(bgcolor="white", font_size=12, align="left")
    )
    fig.show()
else:
    print("資料不足，無法繪圖。")

## 向量資料庫視覺化 (3D t-SNE)
利用 t-SNE 將高維向量投影到 3D 空間。

In [ ]:
if len(embeddings_data) > 0:
    print("正在進行 3D 降維運算 (t-SNE)...")
    tsne_3d = TSNE(n_components=3, perplexity=min(30, len(embeddings_data)-1), init='pca', learning_rate='auto', random_state=42)
    embeddings_3d = tsne_3d.fit_transform(embeddings_data)

    fig_3d = go.Figure()
    for i, source in enumerate(unique_sources):
        mask = np.array([s == source for s in sources])
        fig_3d.add_trace(go.Scatter3d(
            x=embeddings_3d[mask, 0],
            y=embeddings_3d[mask, 1],
            z=embeddings_3d[mask, 2],
            mode='markers',
            name=source,
            text=[f"<b>File:</b> {source}<br><b>Content:</b><br>{wrap_text(documents[j][:200])}..." for j in np.where(mask)[0]],
            hoverinfo='text',
            marker=dict(size=5, opacity=0.8)
        ))
    
    fig_3d.update_layout(
        title=f"Vector Database 3D Visualization (Total Chunks: {len(embeddings_data)})", 
        width=1000, height=800,
        hoverlabel=dict(bgcolor="white", font_size=12, align="left")
    )
    fig_3d.show()
else:
    print("資料不足，無法繪圖。")